# InferOps Week 4: Continuous batching with vLLM

**Question:** how many users can one T4 serve within the SLO (TTFT <= 1 s, TPOT <= 100 ms), and what does batching buy you?

| Part | What runs | Why |
|---|---|---|
| A | our HF engine, concurrency 1-8 | the naive baseline: one request at a time, everyone else queues |
| B | vLLM FP16 eager, FP16, AWQ, concurrency 1-128 | clean vLLM numbers (week 3 ran with `--enforce-eager` by mistake), and what CUDA graphs are worth |
| C | `vllm bench serve` vs our load generator | can we trust our own benchmark tool? |
| D | vLLM FP16 with `--max-num-seqs` 8 and 32 | the scheduler knob that trades TTFT against TPOT |

Every config gets the same speed test. Quality isn't re-measured: batching and CUDA graphs don't change the weights
(week 3 measured GSM8K for FP16 and AWQ).

**Predict before you run:** at concurrency 8, how long will the 8th request in line wait for its first token on our
HF engine? (Week 3: one request takes ~5.8 s.) And how much higher will vLLM's TPOT be at 64 users than at 1?

**Setup:** Kaggle, GPU T4 x2, Internet on. **Budget:** about 60-75 GPU minutes. Stop the session when done.

## 1. Get the code

In [ ]:
import os
%cd /kaggle/working
if os.path.exists("inferops"):
    !git -C inferops pull
else:
    !git clone https://github.com/saideepa05/inferops.git
%cd /kaggle/working/inferops
!pip install -q -e ".[hf,dev]"
!python -m pytest -q 2>&1 | tail -1

import torch, transformers
print("torch", torch.__version__, "| transformers", transformers.__version__,
      "| GPUs:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

## 2. Experiment configuration

Each concurrency level runs `max(16, 4 x concurrency)` requests, so every simulated user sends at least 4 requests.
With fewer, the level is mostly ramp-up and ramp-down and the averages lie.

In [ ]:
MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"
AWQ_ID = "Qwen/Qwen2.5-3B-Instruct-AWQ"
OUT_DIR, RAW_DIR = "results/week4", "/kaggle/working/raw"
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(RAW_DIR, exist_ok=True)

DATASET, MAX_TOKENS = "medium", 128      # ~330-token prompts, fixed 128-token outputs
SLO_TTFT_MS, SLO_TPOT_MS = 1000, 100

HF_LEVELS = [1, 2, 4, 8]                 # serial engine: TTFT is already far past the SLO at 8
VLLM_LEVELS = [1, 4, 8, 16, 32, 64, 128]
MIN_REQUESTS, WAVES = 16, 4

## 3. Helpers

In [ ]:
import json, re, subprocess, time, urllib.request

def start_server(cmd, env_extra, port, log_path, timeout_s=900):
    """Launch a server in the background and wait until /health answers. Returns (process, startup seconds)."""
    log = open(log_path, "w")
    proc = subprocess.Popen(cmd, env=os.environ | env_extra, stdout=log, stderr=subprocess.STDOUT)
    start = time.time()
    while time.time() - start < timeout_s:
        if proc.poll() is not None:
            raise RuntimeError(f"Server exited early. Last log lines:\n{open(log_path).read()[-3000:]}")
        try:
            urllib.request.urlopen(f"http://127.0.0.1:{port}/health", timeout=2)
            startup_s = time.time() - start
            print(f"Ready after {startup_s:.0f}s on port {port}")
            return proc, startup_s
        except Exception:
            time.sleep(3)
    proc.terminate()
    raise TimeoutError(f"Server not ready after {timeout_s}s; see {log_path}")

def stop_server(proc):
    proc.terminate()
    try:
        proc.wait(timeout=60)
    except subprocess.TimeoutExpired:
        proc.kill()
    time.sleep(5)  # let the driver release GPU memory before the next server

def gpu_mem_mib(index):
    out = subprocess.run(["nvidia-smi", f"--id={index}", "--query-gpu=memory.used",
                          "--format=csv,noheader,nounits"], capture_output=True, text=True)
    return float(out.stdout.strip())

def loadgen(port, concurrency, requests, out, max_tokens=MAX_TOKENS):
    !python -m bench.loadgen --url http://127.0.0.1:{port} --dataset {DATASET} --concurrency {concurrency} --num-requests {requests} --max-tokens {max_tokens} --slo-ttft-ms {SLO_TTFT_MS} --slo-tpot-ms {SLO_TPOT_MS} --out {out}

def run_speed(label, port, levels):
    """One loadgen run per level (so each gets its own request count), merged into {label}_speed.json."""
    merged = None
    for c in levels:
        raw = f"{RAW_DIR}/{label}_c{c}.json"
        loadgen(port, c, max(MIN_REQUESTS, WAVES * c), raw)
        run = json.load(open(raw))
        merged = merged or {"meta": run["meta"] | {"label": label, "num_requests": "max(16, 4 x concurrency)"},
                            "levels": []}
        merged["levels"] += run["levels"]
    json.dump(merged, open(f"{OUT_DIR}/{label}_speed.json", "w"), indent=2)

def warmup(port):
    """Throwaway pass at the highest concurrency. Triton compiles some attention kernels the first
    time it sees a batch shape (week 3 log: 'JIT compilation during inference'); that must not land in a measurement."""
    loadgen(port, max(VLLM_LEVELS), max(VLLM_LEVELS), f"{RAW_DIR}/warmup.json", max_tokens=16)

def save_meta(label, **fields):
    json.dump(fields, open(f"{OUT_DIR}/{label}_meta.json", "w"), indent=2)
    print(label, fields)

# Part A: the naive baseline

Our HF engine holds a semaphore of size 1: one request runs, the rest wait in line. Each request still decodes
at full single-user speed, so **TPOT stays flat while TTFT grows with the queue.** This is the shape to compare
against vLLM.

In [ ]:
label = "hf-fp16"
proc, startup_s = start_server(
    ["python", "-m", "uvicorn", "inferops.server:app", "--port", "8000", "--log-level", "warning"],
    {"CUDA_VISIBLE_DEVICES": "0", "INFEROPS_BACKEND": "hf", "INFEROPS_MODEL_ID": MODEL_ID,
     "INFEROPS_DEVICE": "cuda", "INFEROPS_DTYPE": "float16"},
    port=8000, log_path=f"{label}.log")
try:
    save_meta(label, engine="hf", precision="fp16", model=MODEL_ID, startup_s=startup_s,
              loaded_mem_gb=gpu_mem_mib(0) / 1024)
    run_speed(label, 8000, HF_LEVELS)
finally:
    stop_server(proc)

# Part B: vLLM

Same separate Python 3.12 environment as week 3 (3-6 minutes to install).

In [ ]:
VLLM_ENV = "/kaggle/working/vllm-env"
!pip install -q uv
!uv venv -q {VLLM_ENV} --python 3.12
!uv pip install -q --python {VLLM_ENV}/bin/python vllm
!{VLLM_ENV}/bin/python -c "import vllm, torch; print('vllm', vllm.__version__, '| torch', torch.__version__)"

**Prefix caching is off for the whole week.** vLLM turns it on by default, and our load generator sends the same
prompts at every concurrency level. From the second level on, every prompt is already cached, prefill is skipped,
and TTFT looks better than it is. That happened in week 3. Week 5 measures prefix caching on purpose.

**What CUDA graphs do:** in eager mode, Python launches every kernel of every layer on every decode step, and on a
small model at low batch size the GPU spends much of each step waiting for the CPU. A CUDA graph records the whole
step once at startup and replays it with a single launch. `vllm-fp16-eager` vs `vllm-fp16` measures what that's worth.

**Predict:** will the eager vs graphs gap be bigger at concurrency 1 or at 128? Why?

In [ ]:
VLLM_ARGS = ["--dtype", "half", "--max-model-len", "4096", "--gpu-memory-utilization", "0.90",
             "--no-enable-prefix-caching"]

def start_vllm(model, label, extra_args=()):
    return start_server([f"{VLLM_ENV}/bin/vllm", "serve", model, "--port", "8001", *VLLM_ARGS, *extra_args],
                        {"CUDA_VISIBLE_DEVICES": "1"}, port=8001, log_path=f"{label}.log", timeout_s=1200)

def parse_vllm_log(path):
    text = open(path).read()
    grab = lambda pattern: (m := re.search(pattern, text)) and float(m.group(1).replace(",", ""))
    return {"weights_gb": grab(r"(?:weights take|loading took) ([\d.]+) ?GiB"),
            "kv_cache_tokens": grab(r"KV cache size: ([\d,]+) tokens"),
            "max_concurrency_at_4096": grab(r"Maximum concurrency for [\d,]+ tokens per request: ([\d.]+)")}

def vllm_meta(label, model, extra_args, startup_s):
    seqs = extra_args[extra_args.index("--max-num-seqs") + 1] if "--max-num-seqs" in extra_args else "default"
    save_meta(label, engine="vllm", precision="awq" if "AWQ" in model else "fp16", model=model,
              enforce_eager="--enforce-eager" in extra_args, prefix_caching=False, max_num_seqs=seqs,
              startup_s=startup_s, **parse_vllm_log(f"{label}.log"))

def run_vllm(label, model, extra_args, levels):
    print(f"\n===== {label} =====")
    try:
        proc, startup_s = start_vllm(model, label, extra_args)
    except Exception as exc:
        print(f"{label} failed to start: {str(exc)[-1500:]}")
        return
    try:
        vllm_meta(label, model, list(extra_args), startup_s)
        warmup(8001)
        run_speed(label, 8001, levels)
    finally:
        stop_server(proc)

for label, model, extra_args in [("vllm-fp16-eager", MODEL_ID, ["--enforce-eager"]),
                                 ("vllm-fp16", MODEL_ID, []),
                                 ("vllm-awq", AWQ_ID, [])]:
    run_vllm(label, model, extra_args, VLLM_LEVELS)

**Look before moving on:** compare `startup_s` in the three meta files. Graph capture and torch.compile cost
startup time. When does that matter in production? (Think autoscaling.)

# Part C: can we trust our load generator?

vLLM ships its own benchmark, `vllm bench serve`. Point both tools at the same server, with the same input and output
lengths and the same concurrency, and compare. If they agree, every number in this project so far stands.

Differences you should expect:
- `vllm bench` uses random token prompts, we use text. Content doesn't matter for speed at fixed lengths with prefix caching off.
- With `--max-concurrency` and no request rate, it sends a new request as soon as one finishes: the same closed loop as ours.
- Both define TPOT as `(e2e - TTFT) / (output tokens - 1)`.

Agreement within ~5% means the tools measure the same thing. A bigger gap is worth explaining before you trust either.

In [ ]:
from transformers import AutoTokenizer
from bench.prompts import build_dataset

# Match vLLM's random prompts to ours: our prompts' mean token count minus what the chat template adds
# (vLLM's chat backend applies the template on the server, on top of --random-input-len).
tok = AutoTokenizer.from_pretrained(MODEL_ID)
n_tokens = lambda msgs: len(tok.apply_chat_template(msgs, add_generation_prompt=True, tokenize=True))
template = n_tokens([{"role": "user", "content": ""}])
our_len = sum(map(n_tokens, build_dataset(DATASET, n=64))) / 64
IN_LEN = round(our_len - template)
print(f"our prompts: {our_len:.0f} tokens incl. {template} of chat template -> --random-input-len {IN_LEN}")

CHECK_LEVELS = [1, 16, 64]
proc, _ = start_vllm(MODEL_ID, "validate")
try:
    warmup(8001)
    run_speed("validate-loadgen", 8001, CHECK_LEVELS)
    for c in CHECK_LEVELS:
        n = max(MIN_REQUESTS, WAVES * c)
        !{VLLM_ENV}/bin/vllm bench serve --backend openai-chat --base-url http://127.0.0.1:8001 --endpoint /v1/chat/completions --model {MODEL_ID} --dataset-name random --random-input-len {IN_LEN} --random-output-len {MAX_TOKENS} --num-prompts {n} --max-concurrency {c} --ignore-eos --percentile-metrics ttft,tpot,itl,e2el --metric-percentiles 50,95,99 --goodput ttft:{SLO_TTFT_MS} tpot:{SLO_TPOT_MS} --seed 0 --save-result --result-dir {OUT_DIR} --result-filename vllmbench_c{c}.json
finally:
    stop_server(proc)

In [ ]:
import pandas as pd

ours = {lv["concurrency"]: lv for lv in json.load(open(f"{OUT_DIR}/validate-loadgen_speed.json"))["levels"]}
rows = []
for c in CHECK_LEVELS:
    p = f"{OUT_DIR}/vllmbench_c{c}.json"
    if not os.path.exists(p):
        print(f"missing {p}: check the vllm bench output above")
        continue
    vb, us = json.load(open(p)), ours[c]
    pairs = {"ttft_p50_ms": (us["ttft_s"]["p50"] * 1000, vb.get("median_ttft_ms")),
             "ttft_p95_ms": (us["ttft_s"]["p95"] * 1000, vb.get("p95_ttft_ms")),
             "tpot_p50_ms": (us["tpot_s"]["p50"] * 1000, vb.get("median_tpot_ms")),
             "out_tok_s": (us["output_tok_s"], vb.get("output_throughput")),
             "goodput_rps": (us["goodput_rps"], vb.get("request_goodput"))}
    for metric, (a, b) in pairs.items():
        rows.append({"concurrency": c, "metric": metric, "ours": a, "vllm_bench": b,
                     "diff_pct": (a - b) / b * 100 if b else None})
    print(f"c={c}: vllm bench mean input {vb.get('total_input_tokens', 0) / max(vb.get('completed', 1), 1):.0f} tokens")
validation = pd.DataFrame(rows)
validation.round(1)

# Part D: the scheduler knob

`--max-num-seqs` caps how many requests vLLM decodes in one batch (default 256 on this setup). Requests beyond the
cap wait in vLLM's queue, the same way they wait in our HF engine's semaphore.

**Predict:** with `--max-num-seqs 8` and 64 users, what happens to TTFT, TPOT and throughput compared with the
default? Is there a setting that gets *more* goodput than the default?

In [ ]:
for seqs in [8, 32]:
    run_vllm(f"vllm-fp16-seqs{seqs}", MODEL_ID, ["--max-num-seqs", str(seqs)], [16, 64, 128])

## 4. Results tables

In [ ]:
import glob

ORDER = ["hf-fp16", "vllm-fp16-eager", "vllm-fp16", "vllm-awq", "vllm-fp16-seqs8", "vllm-fp16-seqs32"]
metas, levels = {}, []
for label in ORDER:
    if not os.path.exists(p := f"{OUT_DIR}/{label}_speed.json"):
        continue
    metas[label] = json.load(open(f"{OUT_DIR}/{label}_meta.json"))
    for lv in json.load(open(p))["levels"]:
        levels.append({"config": label, "conc": lv["concurrency"], "requests": lv["requests"],
                       "rps": lv["rps"], "out_tok_s": lv["output_tok_s"],
                       "ttft_p50_ms": lv["ttft_s"]["p50"] * 1000, "ttft_p95_ms": lv["ttft_s"]["p95"] * 1000,
                       "tpot_p50_ms": lv["tpot_s"]["p50"] * 1000, "tpot_p95_ms": lv["tpot_s"]["p95"] * 1000,
                       "slo_pct": lv["slo_attainment"] * 100, "goodput_rps": lv["goodput_rps"],
                       "errors": lv["errors"]})
per_level = pd.DataFrame(levels)

# Capacity: the highest concurrency where at least 90% of requests met the SLO.
summary = []
for label, g in per_level.groupby("config", sort=False):
    ok = g[g.slo_pct >= 90]
    best = ok.loc[ok.conc.idxmax()] if len(ok) else None
    summary.append({"config": label, "startup_s": metas[label].get("startup_s"),
                    "kv_cache_tokens": metas[label].get("kv_cache_tokens"),
                    "tpot_c1_ms": g[g.conc == 1].tpot_p50_ms.min() if (g.conc == 1).any() else None,
                    "max_conc_in_slo": best.conc if best is not None else 0,
                    "goodput_rps_there": best.goodput_rps if best is not None else 0.0,
                    "out_tok_s_there": best.out_tok_s if best is not None else 0.0,
                    "peak_out_tok_s": g.out_tok_s.max()})
summary = pd.DataFrame(summary).set_index("config")
display(summary.round(1))
per_level.round(1)

**How to read it:**
- `max_conc_in_slo` is the capacity number: how many simultaneous users this config serves with >= 90% of requests inside the SLO.
- `goodput_rps_there` is requests per second *that met the SLO* at that load. Raw throughput past that point is load nobody would accept.
- For HF, compare `ttft_p50_ms` across levels with your prediction from the top of the notebook.

## 5. Charts

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import FixedLocator, NullLocator

INK, MUTED, GRID, SURFACE = "#0b0b0b", "#52514e", "#e6e5e0", "#fcfcfb"

def label_line_ends(ax, ends, gap_pt=10):
    """Name each line at its right end, nudging labels apart vertically when lines finish close together."""
    ax.get_ylim()  # force autoscaling so data-to-pixel transforms are final
    to_pt = 72 / ax.figure.dpi
    ys = [ax.transData.transform((x, y))[1] * to_pt for _, x, y in ends]
    placed, last = {}, float("-inf")
    for i in sorted(range(len(ends)), key=ys.__getitem__):
        placed[i] = last = max(ys[i], last + gap_pt)
    for i, (cfg, x, y) in enumerate(ends):
        ax.annotate(cfg, (x, y), textcoords="offset points", xytext=(6, placed[i] - ys[i]),
                    va="center", fontsize=8, color=MUTED)

def plot_sweep(configs, colors, title, path):
    """Four panels against concurrency; one line per config, labelled at its right end."""
    panels = [("ttft_p95_ms", "TTFT p95 (ms, log scale)", SLO_TTFT_MS),
              ("tpot_p50_ms", "TPOT p50 (ms)", SLO_TPOT_MS),
              ("out_tok_s", "Output tokens/s", None),
              ("goodput_rps", "Goodput (requests/s within SLO)", None)]
    configs = [c for c in configs if c in set(per_level.config)]
    fig, axes = plt.subplots(1, 4, figsize=(19, 4.8), facecolor=SURFACE)
    for ax, (col, name, slo) in zip(axes, panels):
        ax.set_facecolor(SURFACE)
        ends = []
        for cfg in configs:
            g = per_level[per_level.config == cfg].sort_values("conc")
            ax.plot(g.conc, g[col], color=colors[cfg], lw=2, marker="o", ms=5,
                    markeredgecolor=SURFACE, markeredgewidth=1.5, label=cfg)
            ends.append((cfg, g.conc.iloc[-1], g[col].iloc[-1]))
        if slo:
            ax.axhline(slo, color=MUTED, lw=1, ls="--")
            ax.annotate("SLO", (1, slo), textcoords="offset points", xytext=(2, 3), fontsize=8, color=MUTED)
        if col == "ttft_p95_ms":
            ax.set_yscale("log")
        else:
            ax.set_ylim(bottom=0)
        ax.set_xscale("log", base=2)
        concs = sorted(per_level[per_level.config.isin(configs)].conc.unique())
        ax.xaxis.set_major_locator(FixedLocator(concs))
        ax.xaxis.set_minor_locator(NullLocator())
        ax.set_xticklabels([str(c) for c in concs])
        ax.set_xlim(right=max(concs) * 2.2)  # room for the end labels
        label_line_ends(ax, ends)
        ax.set_xlabel("concurrent users", color=MUTED)
        ax.set_title(name, color=INK, loc="left", fontsize=11)
        ax.tick_params(colors=MUTED)
        ax.grid(True, axis="y", color=GRID, lw=0.8)
        ax.set_axisbelow(True)
        for side in ("top", "right"):
            ax.spines[side].set_visible(False)
        for side in ("left", "bottom"):
            ax.spines[side].set_color("#c9c8c2")
    fig.legend(handles=[plt.Line2D([], [], color=colors[c], lw=2, marker="o") for c in configs],
               labels=configs, frameon=False, ncol=len(configs), loc="upper right")
    fig.suptitle(title, color=INK, x=0.01, ha="left", fontsize=13)
    fig.tight_layout()
    fig.subplots_adjust(top=0.82)
    fig.savefig(path, dpi=150, facecolor=SURFACE)
    plt.show()

# Colour = config identity, fixed across charts (validated categorical palette, slots 1-4).
plot_sweep(["hf-fp16", "vllm-fp16-eager", "vllm-fp16", "vllm-awq"],
           {"hf-fp16": "#2a78d6", "vllm-fp16": "#eb6834", "vllm-awq": "#1baf7a", "vllm-fp16-eager": "#eda100"},
           f"Batching on a Tesla T4: {MODEL_ID}, medium prompts, 128 output tokens", f"{OUT_DIR}/week4_batching.png")
# max-num-seqs is a magnitude, so one hue from light (small cap) to dark (default).
plot_sweep(["vllm-fp16-seqs8", "vllm-fp16-seqs32", "vllm-fp16"],
           {"vllm-fp16-seqs8": "#86b6ef", "vllm-fp16-seqs32": "#2a78d6", "vllm-fp16": "#104281"},
           "vLLM FP16: --max-num-seqs 8 / 32 / default", f"{OUT_DIR}/week4_scheduler.png")

## 6. Write the report and package the results

In [ ]:
from datetime import datetime, timezone

def md_table(frame, index=True):
    frame = frame.reset_index() if index else frame
    lines = ["| " + " | ".join(frame.columns) + " |", "|" + "---|" * len(frame.columns)]
    for _, row in frame.iterrows():
        cells = ["" if pd.isna(v) else f"{v:.1f}" if isinstance(v, float) else str(v) for v in row]
        lines.append("| " + " | ".join(cells) + " |")
    return "\n".join(lines)

vllm_version = subprocess.run([f"{VLLM_ENV}/bin/python", "-c", "import vllm; print(vllm.__version__)"],
                              capture_output=True, text=True).stdout.strip() or "not installed"
report = f"""# Week 4 report: continuous batching on a Tesla T4

Generated {datetime.now(timezone.utc):%Y-%m-%d %H:%M UTC}

## Environment
| | |
|---|---|
| GPU | {torch.cuda.get_device_name(0)} |
| Model | {MODEL_ID} (FP16) and {AWQ_ID} |
| torch / transformers (HF engine) | {torch.__version__} / {transformers.__version__} |
| vLLM | {vllm_version}, args: {" ".join(VLLM_ARGS)} (CUDA graphs on unless the config says eager) |

## Workload
- `{DATASET}` prompts, {MAX_TOKENS} output tokens (ignore_eos), temperature 0, closed loop.
- Each level runs max({MIN_REQUESTS}, {WAVES} x concurrency) requests. vLLM gets a warm-up pass at concurrency {max(VLLM_LEVELS)} first.
- SLO: TTFT <= {SLO_TTFT_MS} ms and TPOT <= {SLO_TPOT_MS} ms per request. Capacity = highest concurrency with >= 90% of requests inside it.

## Capacity summary
{md_table(summary)}

## All levels
{md_table(per_level, index=False)}

![batching](week4_batching.png)

![scheduler](week4_scheduler.png)

## Load generator vs `vllm bench serve`
Same vLLM FP16 server, `--random-input-len {IN_LEN}` to match our prompts.

{md_table(validation, index=False) if len(validation) else "_vllm bench results missing_"}

## Findings
_Write these yourself. See the questions at the end of the notebook._

## Decision
_What would you set `--max-num-seqs` to for this workload, and how many T4s would 200 concurrent users need?_
"""
open(f"{OUT_DIR}/report.md", "w").write(report)
!cp *.log {OUT_DIR}/ 2>/dev/null; true
!cd /kaggle/working/inferops && zip -qr /kaggle/working/week4_results.zip {OUT_DIR}
print(report)
print("Download week4_results.zip from the Output panel (right side, /kaggle/working).")

## 7. Questions to answer in your Findings

1. **Naive baseline:** how did HF's TTFT grow with concurrency? Derive it: with one request at a time taking T seconds,
   what is the mean wait at concurrency c? Does the measurement match?
2. **Batching:** why does vLLM's TPOT barely move from 1 to 16 users? Use week 3's idea: what does each decode step
   have to read from memory, and how much of that is shared by every request in the batch?
3. **CUDA graphs:** how big was the eager vs graphs gap at concurrency 1, and at 128? Explain why it shrinks
   (or doesn't) as the batch grows.
4. **AWQ vs FP16 under load:** week 3 showed AWQ faster for one user. Does that hold at 128 users? Where does the
   AWQ kernel's extra dequantization work start to show?
5. **Validation:** did our load generator agree with `vllm bench serve`? Explain any gap above 5%.
6. **Scheduler:** what did `--max-num-seqs 8` do to TTFT and TPOT at 64 users? Draw the tradeoff in one sentence.
7. **Capacity:** using `max_conc_in_slo`, how many T4s would 200 concurrent chat users need with each config?
   What assumption about real traffic makes that number optimistic? (Week 7 tests it with open-loop traffic.)